# Static (training-free) experiments of arXiv:2608.19748: Figs 2, 3, 5, 6, 7 and App. B.3

Runs on CPU from `static_data.npz` (regenerate it with `python scripts/export_static_data.py`) and writes the
figures to `figs/`.

In [ ]:
import os
os.environ["SOURCE_DATE_EPOCH"] = "0"

import matplotlib.pyplot as plt
from matplotlib import patches, transforms
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import rankdata

import sys
from paper_style import apply_style, style_legend
apply_style()

# Plot 1 palette (quantile agreement)
C1_A = '#546E7A'   # blue-gray
C1_B = '#BF7B50'   # terracotta
C1_C = '#8AA4B0'   # mist blue
C1_D = '#D8A07E'   # sand terracotta

# Plot 2 palette (selection quality)
C2_A = "#8866A8"   # muted purple
C2_B = "#5CA077"   # sage green

def add_quantile_guide_bars(
    ax,
    guide_ticks,
    *,
    x_to_quantile,
    bottom_color,
    top_color,
    bar_width=3,
    bar_y=-0.40,
    bar_height=0.15,
    alpha=0.7,
    border_color="black",
    border_linewidth=0.8,
):
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    xmin, xmax = ax.get_xlim()

    for x in guide_ticks:
        if x < xmin or x > xmax:
            continue

        q = float(np.clip(x_to_quantile(x), 0.0, 1.0))
        q = min(max(q, 0.00), 1.0)

        # Bottom region: [0, q]
        bottom_h = bar_height * q

        # Top region: [1-q, 1]
        top_y = bar_y + bar_height * (1.0 - q)
        top_h = bar_height * q

        # Filled bottom block
        if bottom_h > 0:
            ax.add_patch(
                patches.Rectangle(
                    (x - bar_width / 2.0, bar_y),
                    bar_width,
                    bottom_h,
                    transform=trans,
                    facecolor=bottom_color,
                    edgecolor="none",
                    alpha=alpha,
                    clip_on=False,
                    zorder=2,
                )
            )

        # Filled top block
        if top_h > 0:
            ax.add_patch(
                patches.Rectangle(
                    (x - bar_width / 2.0, top_y),
                    bar_width,
                    top_h,
                    transform=trans,
                    facecolor=top_color,
                    edgecolor="none",
                    alpha=alpha,
                    clip_on=False,
                    zorder=2,
                )
            )

        # Outer border around the whole guide bar
        ax.add_patch(
            patches.Rectangle(
                (x - bar_width / 2.0, bar_y),
                bar_width,
                bar_height,
                transform=trans,
                facecolor="none",
                edgecolor=border_color,
                linewidth=border_linewidth,
                clip_on=False,
                zorder=3,
            )
        )

import matplotlib
matplotlib.rcParams["svg.hashsalt"] = "tup-release"


In [ ]:
from pathlib import Path
DATA = Path("static_data.npz")
if not DATA.exists():
    raise FileNotFoundError("notebooks/static_data.npz is missing; regenerate it with: python scripts/export_static_data.py")
_npz = np.load(DATA)
RM_LABELS = {"armorm": "ArmoRM", "skywork_v2_llama": "Skywork-Llama", "skywork_v2_qwen": "Skywork-Qwen"}
train_rm = "armorm"
# One entry per published file; identical to PANELS in scripts/export_static_data.py.
PANELS = [
    {"figure": "fig2", "plot": "region", "dataset": "ultrafeedback", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_llama"], "file_label": "Skywork-Llama"},
    {"figure": "fig2", "plot": "benefit", "dataset": "ultrafeedback", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_llama"], "file_label": "Skywork-Llama"},
    {"figure": "fig6", "plot": "region", "dataset": "ultrafeedback", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_qwen"], "file_label": "Skywork-Qwen"},
    {"figure": "fig6", "plot": "region", "dataset": "magpie", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_llama"], "file_label": "Skywork-Llama"},
    {"figure": "fig6", "plot": "region", "dataset": "magpie", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_qwen"], "file_label": "Skywork-Qwen"},
    {"figure": "fig7", "plot": "benefit", "dataset": "ultrafeedback", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_qwen"], "file_label": "Skywork-Qwen"},
    {"figure": "fig7", "plot": "benefit", "dataset": "magpie", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_llama"], "file_label": "Skywork-Llama"},
    {"figure": "fig7", "plot": "benefit", "dataset": "magpie", "variant": "random", "source": "merged",
     "eval_rms": ["skywork_v2_qwen"], "file_label": "Skywork-Qwen"},
]

def load_panel(panel):
    global rewards, ranks, eval_rms, eval_labels, selected_eval_label, SELECTED_DATASET
    prefix = f'{panel["dataset"]}/{panel["variant"]}/{panel["source"]}'
    eval_rms = list(panel["eval_rms"])
    eval_labels = [RM_LABELS[rm] for rm in eval_rms]
    selected_eval_label = eval_labels[0]  # set by the region-plot code and read by the benefit-plot code
    assert selected_eval_label == panel["file_label"]
    rewards = {rm: _npz[f"{prefix}/{rm}"] for rm in [train_rm, *eval_rms]}
    n_prompts, n_completions = rewards[train_rm].shape
    ranks = {rm: np.array([rankdata(rewards[rm][p], method="average") for p in range(n_prompts)]) / n_completions
             for rm in [train_rm, *eval_rms]}
    SELECTED_DATASET = panel["dataset"]
    print(f'{panel["figure"]} {panel["plot"]}: {prefix} eval_rms={eval_rms} ({n_prompts} prompts x {n_completions})')
os.makedirs("figs", exist_ok=True)


In [ ]:
def plot_region_agreement():
    from matplotlib.ticker import PercentFormatter, MultipleLocator
    from matplotlib.transforms import offset_copy


    selected_eval_rm = eval_rms[0]      # or a specific RM name
    selected_eval_label = eval_labels[0]  # optional, for legend/title


    # ── Plot 1: Bottom vs. Top quantile placement agreement ──────────────
    thresholds = np.arange(0.02, 1.02, 0.02)  # 2% to 100%
    bottom_agree = np.zeros(len(thresholds))
    top_agree = np.zeros(len(thresholds))

    for i, q in enumerate(thresholds):
        arm_bot = ranks[train_rm] <= q
        arm_top = ranks[train_rm] > (1 - q)
        ba, ta = 0.0, 0.0
        for rm in eval_rms:
            ev_bot = ranks[rm] <= q
            ev_top = ranks[rm] > (1 - q)
            ba += (arm_bot & ev_bot).sum() / arm_bot.sum()
            ta += (arm_top & ev_top).sum() / arm_top.sum()
        bottom_agree[i] = ba / len(eval_rms)
        top_agree[i] = ta / len(eval_rms)

    fig, ax = plt.subplots(figsize=(4, 3))
    fig.subplots_adjust(bottom=0.28)


    ax.plot(thresholds, bottom_agree,
            lw=2, color=C1_A, marker='o', ms=3, markevery=5,
            label=rf'{selected_eval_label}: bottom $p$')
    ax.plot(thresholds, top_agree,
            lw=2, color=C1_B, marker='s', ms=3, markevery=5,
            label=rf'{selected_eval_label}: top $p$')


    # ax.plot(thresholds, bottom_agree,
    #         lw=2, color=C1_A, marker='o', ms=3, markevery=5,
    #         label=r'Agreement on bottom $p$')
    # ax.plot(thresholds, top_agree,
    #         lw=2, color=C1_B, marker='s', ms=3, markevery=5,
    #         label=r'Agreement on top $p$')

    plot1_guide_ticks = [0, 20, 40, 60, 80, 100]
    # add_quantile_guide_bars(ax, plot1_guide_ticks,
    #     x_to_quantile=lambda x: x / 100.0,
    #     bottom_color=C1_A,
    #     top_color=C1_B,
    # )

    # Shade the gap between curves
    ax.fill_between(thresholds, bottom_agree, top_agree,
                    alpha=0.10, color='#575757', linewidth=0)

    # # Dangerous zone: shade region beyond 50%
    # ax.axvline(50, color='#B84040', lw=1.2, ls='--', zorder=3)
    # ax.axvspan(50, 100, alpha=0.06, color='#B84040', zorder=0)
    # ax.text(75, ax.get_ylim()[0] + 0.03, 'Rank inversion zone',
    #         ha='center', va='bottom', fontsize=11, color='#B84040')

    # Annotate difference with spanning arrows
    for idx in [0]:
        x = thresholds[idx]
        y_top = max(top_agree[idx], bottom_agree[idx])
        y_bot = min(top_agree[idx], bottom_agree[idx])
        diff = y_top - y_bot
        mid_y = (y_top + y_bot) / 2

        # Text label
        label = ax.annotate(f'{diff:.2f}',
                            xy=(x, mid_y), ha='center', va='center',
                            fontsize=11, color='#555555',
                            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#555555', alpha=0.8,linewidth=0.8))

        # The label box has a fixed size in points while the gap does not, so start each arrow
        # at the box edge (measured from the drawn box) instead of at a fraction of the gap.
        fig.canvas.draw()
        box_half_h = label.get_bbox_patch().get_window_extent().height / 2 * 72 / fig.dpi  # points
        for sign, y_end in [(1, y_top), (-1, y_bot)]:
            arrow_start = offset_copy(ax.transData, fig=fig, y=sign * box_half_h, units='points')
            ax.annotate('', xy=(x, y_end), xytext=(x, mid_y), textcoords=arrow_start,
                        arrowprops=dict(arrowstyle='->', color='#555555', lw=1,
                                        shrinkA=0, shrinkB=1))

    ax.xaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))

    # ax.set_xlabel('$p$', fontsize=12)
    # ax.set_ylabel('Agreement rate\nwith $RM_B$', fontsize=12)
    # ax.legend(frameon=True, edgecolor='#CCCCCC', fancybox=False, framealpha=0.95, fontsize=11)

    ax.set_xlabel('Selected fraction $p$', fontsize=12)
    ax.set_ylabel(f'Agreement rate\nwith {selected_eval_label}', fontsize=12)

    style_legend(
        ax,
        title='Regions defined by ArmoRM',
        fontsize=8,
        title_fontsize=9,
        loc='lower right',
        corner_radius=0.8,
    )

    sns.despine(ax=ax)

    # add hotizontal lines for the plot's major y-ticks, to make it easier to visually estimate agreement rates
    for y in ax.get_yticks():
        ax.axhline(y, color='#CCCCCC', lw=0.8, ls='--', zorder=0)

    for x in ax.get_xticks():
        ax.axvline(x, color='#CCCCCC', lw=0.8, ls='--', zorder=0)

    ax.set_xlim(-0.05, 0.55)
    ax.set_ylim(0.2, 0.85)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.24)
    fig.savefig(f'figs/region_agreement_bottom_vs_top_{selected_eval_label}_{SELECTED_DATASET}.svg')
    plt.show()


In [ ]:
def plot_benefit_vs_cost():
    # ── Plot: Benefit vs. Cost of larger λ ───────────────────────────────
    # Line 1 (cost):    Fraction of eval-RM top completions discarded by train-RM
    # Line 2 (benefit): Fraction of eval-RM bottom completions retained by train-RM
    # Computed for the eval RM(s) in eval_rms (a single RM, set per panel by load_panel).

    lambdas = np.arange(0.00, 1.02, 0.02)
    thresh = 0.25  # eval-RM "good" = top `thresh` fraction, "bad" = bottom `thresh` fraction (also used in the legend)
    discard_purity = []
    top1_coverage = []

    for i, lam in enumerate(lambdas):
        discarded = ranks[train_rm] <= lam          # (n_prompts, n_completions)
        retained  = ~discarded
    
        dp_acc, tc = 0.0, 0.0
        for rm in eval_rms:
            ev_good = ranks[rm] > 1 - thresh
            ev_bad = ranks[rm] <= thresh
            dp_acc += (discarded & ev_good).sum() / ev_good.sum()
            tc += (retained & ev_bad).sum() / ev_bad.sum()
        discard_purity.append(dp_acc / len(eval_rms))
        top1_coverage.append(tc / len(eval_rms))


    # ── Figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(5, 3))

    # ax.plot(lambdas * 100, discard_purity,
    #         lw=2, color=C2_A, marker='o', ms=3, markevery=5,
    #         label='$P(\\text{Good completions discarded})$')
    # ax.plot(lambdas * 100, top1_coverage,
    #         lw=2, color=C2_B, marker='s', ms=3, markevery=5,
    #         label='$P(\\text{Bad completions retained})$')

    eval_label_tex = selected_eval_label.replace("_", r"\_")

    # ax.plot(
    #     lambdas, discard_purity,
    #     lw=2, color=C2_A, marker='o', ms=3, markevery=5,
    #     label=rf'$P(\mathrm{{WR}}_{{\mathrm{{{eval_label_tex}}}}} \geq 0.75 \mid \mathrm{{WR}}_{{\mathrm{{ArmoRM}}}} \leq \lambda)$'
    # )

    # ax.plot(
    #     lambdas, top1_coverage,
    #     lw=2, color=C2_B, marker='s', ms=3, markevery=5,
    #     label=rf'$P(\mathrm{{WR}}_{{\mathrm{{{eval_label_tex}}}}} \leq 0.25 \mid \mathrm{{WR}}_{{\mathrm{{ArmoRM}}}} > \lambda)$'
    # )

    ax.plot(
        lambdas, discard_purity,
        lw=2, color=C2_A, marker='o', ms=3, markevery=5,
        label=rf'$P(\mathrm{{WR}}_{{\mathrm{{ArmoRM}}}} \leq \lambda \mid \mathrm{{WR}}_{{\mathrm{{{eval_label_tex}}}}} \geq {1 - thresh:g})$'
    )

    ax.plot(
        lambdas, top1_coverage,
        lw=2, color=C2_B, marker='s', ms=3, markevery=5,
        label=rf'$P(\mathrm{{WR}}_{{\mathrm{{ArmoRM}}}} > \lambda \mid \mathrm{{WR}}_{{\mathrm{{{eval_label_tex}}}}} \leq {thresh:g})$'
    )

    # Shade the gap between curves
    ax.fill_between(lambdas, discard_purity, top1_coverage,
                    alpha=0.10, color="#575757", linewidth=0)

    # Crossover point
    crossover_idx = np.argmin(np.abs(np.array(top1_coverage) - np.array(discard_purity)))
    crossover = lambdas[crossover_idx]
    ax.axvline(crossover, color="#B84040", lw=1.2, ls='--', zorder=3)

    # Annotate crossover with arrow
    cross_y = discard_purity[crossover_idx]
    ax.annotate(
        f'crossover $\\approx$ {crossover:.2f}',
        xy=(crossover, cross_y),
        xytext=(crossover + 0.03, cross_y + 0.30),
        fontsize=10,
        color='#B84040',
        va='center',
        arrowprops=dict(
            arrowstyle='->',
            color='#B84040',
            lw=1.0,
            shrinkA=0,
            shrinkB=3
        ),
        bbox=dict(
            boxstyle='round,pad=0.25',
            fc='white',       # fill color
            ec='#B84040',     # edge color
            lw=1.0,
            alpha=0.95
        )
    )

    # Annotate gap at a representative point (like plot 1)
    ann_idx = 7  # lambda ≈ 14%
    x_ann = lambdas[ann_idx]
    y_top = max(top1_coverage[ann_idx], discard_purity[ann_idx])
    y_bot = min(top1_coverage[ann_idx], discard_purity[ann_idx])
    diff = y_top - y_bot
    mid_y = (y_top + y_bot) / 2
    pad = diff * 0.15

    # add hotizontal lines for the plot's major y-ticks, to make it easier to visually estimate agreement rates
    for y in ax.get_yticks():
        ax.axhline(y, color='#CCCCCC', lw=0.8, ls='--', zorder=0)


    ax.set_xlabel('Truncation threshold $\\lambda$', fontsize=12)
    ax.set_ylabel('Probability', fontsize=12)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    style_legend(
        ax,
        loc='upper center',
        fontsize=10,
        corner_radius=0.8,
    )
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(f'figs/benefit_vs_cost_lambda_{selected_eval_label}_{SELECTED_DATASET}.svg')
    plt.show()


In [ ]:
PLOTS = {"region": plot_region_agreement, "benefit": plot_benefit_vs_cost}
for panel in PANELS:
    load_panel(panel)
    PLOTS[panel["plot"]]()


## Fixed-threshold illustration (Figs 3 and 5)

In [ ]:
# ── Plot 3: Fixed-threshold oracle-utility comparison ────────────────────────
import math
from scipy.optimize import minimize_scalar
from paper_style import ACCENT, BASELINES, style_legend


def qrpo_moments(tau: float):
    """
    First two moments of the QRPO rank density on [0, 1]:

        q_tau(w) = tau * exp(tau * w) / (exp(tau) - 1),   tau > 0,
        q_0(w) = 1.

    Returns:
        mean_w, var_w
    """
    if tau == 0.0:
        return 0.5, 1.0 / 12.0

    et = math.exp(tau)
    mean_w = et / (et - 1.0) - 1.0 / tau
    var_w = 1.0 / (tau * tau) - et / ((et - 1.0) ** 2)
    return mean_w, var_w


def qrpo_utility(s: float, tau: float) -> float:
    """
    Oracle utility under the quadratic oracle profile

        m_s(w) = 1 - (w - s)^2.

    Since U_s = E[m_s(W)] = 1 - E[(W - s)^2], we can compute it from
    the mean and variance of W.
    """
    mean_w, var_w = qrpo_moments(tau)
    risk = (mean_w - s) ** 2 + var_w
    return 1.0 - risk


def qrpo_best_utility(s: float) -> float:
    """Best QRPO oracle utility over tau >= 0."""
    utility_at_zero = qrpo_utility(s, 0.0)
    result = minimize_scalar(
        lambda tau: -qrpo_utility(s, tau),
        bounds=(1e-8, 200.0),
        method='bounded',
        options={'xatol': 1e-8},
    )
    return max(utility_at_zero, -float(result.fun))


def pure_truncation_utility(s_vals: np.ndarray):
    """
    Oracle utility of pure truncation at the fixed threshold t = 1/2.

    Under W ~ Unif[1/2, 1]:
        E[W]   = 3/4
        Var[W] = 1/48.
    """
    mean_w = 0.75
    var_w = 1.0 / 48.0
    return 1.0 - ((mean_w - s_vals) ** 2 + var_w)


# We parameterize the fixed-threshold truncated-soft family by a = 1 / beta,
# so a = 0 is pure truncation and a > 0 adds within-tail upweighting.
def precompute_ours(a_grid: np.ndarray, n_nodes: int = 400):
    """
    Precompute the first two moments of the fixed-threshold family at t = 1/2.

    The density on w in (1/2, 1) is proportional to

        ((w - 1/2) / (3/2 - w))^a,

    where a = 1 / beta.
    """
    x, w_quad = np.polynomial.legendre.leggauss(n_nodes)

    # Change variables: w = 1/2 + u, where u in [0, 1/2].
    u = 0.25 * (x + 1.0)
    jac = 0.25

    log_kernel = np.log(u / (1.0 - u))[None, :]
    kernel = np.exp(a_grid[:, None] * log_kernel)
    base = w_quad[None, :] * jac * kernel

    w = 0.5 + u[None, :]
    partition = np.sum(base, axis=1)
    mean_w = np.sum(base * w, axis=1) / partition
    second_moment_w = np.sum(base * (w ** 2), axis=1) / partition
    return mean_w, second_moment_w


def ours_best_over_grid(
    s_vals: np.ndarray,
    a_grid: np.ndarray,
    mean_w: np.ndarray,
    second_moment_w: np.ndarray,
):
    """
    For each oracle-optimal rank s, pick the best a >= 0 at fixed threshold t = 1/2.
    """
    utilities = 1.0 - (
        second_moment_w[:, None]
        - 2.0 * mean_w[:, None] * s_vals[None, :]
        + (s_vals[None, :] ** 2)
    )
    idx = np.argmax(utilities, axis=0)
    return utilities[idx, np.arange(s_vals.size)], a_grid[idx]


def first_crossing(x: np.ndarray, y: np.ndarray):
    """
    First crossing from positive to non-positive, using linear interpolation.
    """
    for i in range(len(x) - 1):
        if y[i] > 0.0 and y[i + 1] <= 0.0:
            x0, x1 = x[i], x[i + 1]
            y0, y1 = y[i], y[i + 1]
            return x0 - y0 * (x1 - x0) / (y1 - y0)
    return None


s_vals = np.linspace(0.50, 1.0, 240)

# Dense grid over a = 1 / beta. a = 0 is pure truncation.
a_grid = np.concatenate([np.array([0.0]), np.geomspace(1e-4, 40.0, 6000)])
a_grid = np.unique(np.sort(a_grid))

mean_w_ours, second_moment_w_ours = precompute_ours(a_grid, n_nodes=350)
ours_u, best_a = ours_best_over_grid(s_vals, a_grid, mean_w_ours, second_moment_w_ours)
qrpo_u = np.array([qrpo_best_utility(float(s)) for s in s_vals])
pure_trunc_u = pure_truncation_utility(s_vals)

# Analytic threshold from the proposition:
# above s_beta, finite beta improves over pure truncation at t = 1/2.
s_beta = (11.0 / 12.0 - math.log(2.0)) / (1.0 - math.log(2.0))

# Numerical threshold from the paper:
# pure truncation at t = 1/2 stops beating best QRPO here.
s_T = first_crossing(s_vals, pure_trunc_u - qrpo_u)

fill_s_vals = np.unique(np.sort(np.concatenate([s_vals, np.array([s_beta, s_T])])))
fill_qrpo_u = np.interp(fill_s_vals, s_vals, qrpo_u)
fill_pure_trunc_u = pure_truncation_utility(fill_s_vals)
fill_ours_u = np.interp(fill_s_vals, s_vals, ours_u)

fig, ax = plt.subplots(figsize=(6, 3))

ax.plot(fill_s_vals, fill_qrpo_u, color=BASELINES['QRPO'], lw=1.8, label='QRPO')
ax.plot(fill_s_vals, fill_pure_trunc_u, color='#6E6E6E', lw=1.6, ls='--', label='Pure truncation')
ax.plot(fill_s_vals, fill_ours_u, color=ACCENT, lw=2.2, label='Ours')

# Decompose the gain of our fixed-threshold policy into truncation gain and
# additional finite-beta gain.
truncation_gain = fill_pure_trunc_u > fill_qrpo_u
finite_beta_gain_left = (fill_s_vals >= s_beta) & (fill_ours_u > fill_pure_trunc_u)
if s_T is not None:
    finite_beta_gain_left &= fill_s_vals <= s_T

finite_beta_gain_right = None
if s_T is not None:
    finite_beta_gain_right = (fill_s_vals >= s_T) & (fill_ours_u > fill_qrpo_u)

if np.any(truncation_gain):
    ax.fill_between(
        fill_s_vals,
        fill_qrpo_u,
        fill_pure_trunc_u,
        where=truncation_gain,
        color="#EBD8D8",
        alpha=0.38,
        interpolate=True,
        linewidth=0,
        label='Truncation gain',
    )

if np.any(finite_beta_gain_left):
    ax.fill_between(
        fill_s_vals,
        fill_pure_trunc_u,
        fill_ours_u,
        where=finite_beta_gain_left,
        color=ACCENT,
        alpha=0.24,
        interpolate=True,
        linewidth=0,
        label='Finite-$\\beta$ gain',
    )

if finite_beta_gain_right is not None and np.any(finite_beta_gain_right):
    ax.fill_between(
        fill_s_vals,
        fill_qrpo_u,
        fill_ours_u,
        where=finite_beta_gain_right,
        color=ACCENT,
        alpha=0.24,
        interpolate=True,
        linewidth=0,
        label='_nolegend_',
    )



# 1. Move the y-limit calculations UP so we know exactly where the bottom of the graph is
y_min = min(ours_u.min(), qrpo_u.min(), pure_trunc_u.min())
y_max = max(ours_u.max(), qrpo_u.max(), pure_trunc_u.max())
y_pad = 0.06 * (y_max - y_min)
bottom_y = y_min - y_pad

# 2. Calculate the exact y-intercept for s_beta on the pure truncation curve
y_beta = pure_truncation_utility(s_beta)

# 3. Draw a line from the bottom of the plot to the intercept
ax.plot([s_beta, s_beta], [bottom_y, y_beta], color='#6E6E6E', lw=1.0, ls='--')
ax.text(
    s_beta + 0.004,
    bottom_y + 0.022, # Adjusted to use bottom_y
    r'$s_\beta$',
    color='#6E6E6E',
    fontsize=11,
)

if s_T is not None:
    # Calculate the exact y-intercept for s_T
    y_T = pure_truncation_utility(s_T)
    
    ax.plot([s_T, s_T], [bottom_y, y_T], color='#9A9A9A', lw=1.0, ls=':')
    ax.text(
        s_T + 0.004,
        bottom_y + 0.022, 
        r'$s_T$',
        color='#7A7A7A',
        fontsize=11,
    )

ax.set_xlabel(r'Oracle-optimal rank $s$')
ax.set_ylabel(r'Oracle utility $\mathcal{U}_s$')

# 4. Apply the previously calculated limits
ax.set_ylim(bottom_y, y_max + y_pad)
ax.set_xlim(0.5, 1.01)

# ax.set_xlabel(r'Oracle-optimal rank $s$')
ax.set_xlabel(r'Proxy rank where oracle value peaks')
ax.set_ylabel(r'Oracle utility $\mathcal{U}_s$')
ax.set_xlim(s_vals.min(), s_vals.max())

y_min = min(ours_u.min(), qrpo_u.min(), pure_trunc_u.min())
y_max = max(ours_u.max(), qrpo_u.max(), pure_trunc_u.max())
y_pad = 0.06 * (y_max - y_min)
ax.set_ylim(y_min - y_pad, y_max + y_pad)
ax.set_xlim(0.5, 1.01)

# style_legend(
#     ax,
#     ours_label='Ours',
#     fill_edgecolor='#5F6B6C',
#     fill_linewidth=0.8,
#     fill_labels={'Truncation gain', 'Finite-$\\beta$ gain'},
#     corner_radius=0.8,
#     loc='center right',
# )

style_legend(
    ax,
    ours_label='Ours',
    fill_edgecolor='#5F6B6C',
    fill_linewidth=0.8,
    fill_labels={'Truncation gain', 'Finite-$\\beta$ gain'},
    corner_radius=0.8,
    loc='upper left',
    bbox_to_anchor=(1.05, 1.0),
    borderaxespad=0.0,
)
# legend location upper left with bg opacity
# ax.legend(loc='center right', fontsize=9, framealpha=0.95)

sns.despine(ax=ax)
fig.tight_layout()
fig.savefig('figs/fixed_threshold_static.svg')
plt.show()


In [ ]:
# ── Plot 4: Optimal finite beta as a function of s ───────────────────────────
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from paper_style import ACCENT, style_legend


# We reuse the same fixed-threshold family at lambda = 1/2 and parameterize by
# a = 1 / beta. The optimum beta is beta*(s) = 1 / a*(s).
def precompute_ours(a_grid: np.ndarray, n_nodes: int = 400):
    x, w_quad = np.polynomial.legendre.leggauss(n_nodes)

    # w = 1/2 + u, where u in [0, 1/2].
    u = 0.25 * (x + 1.0)
    jac = 0.25

    log_kernel = np.log(u / (1.0 - u))[None, :]
    kernel = np.exp(a_grid[:, None] * log_kernel)
    base = w_quad[None, :] * jac * kernel

    w = 0.5 + u[None, :]
    partition = np.sum(base, axis=1)
    mean_w = np.sum(base * w, axis=1) / partition
    second_moment_w = np.sum(base * (w ** 2), axis=1) / partition
    return mean_w, second_moment_w


def best_a_over_grid(
    s_vals: np.ndarray,
    a_grid: np.ndarray,
    mean_w: np.ndarray,
    second_moment_w: np.ndarray,
):
    utilities = 1.0 - (
        second_moment_w[:, None]
        - 2.0 * mean_w[:, None] * s_vals[None, :]
        + s_vals[None, :] ** 2
    )
    idx = np.argmax(utilities, axis=0)
    return a_grid[idx], utilities[idx, np.arange(s_vals.size)]


s_vals = np.linspace(0.7284, 1.0, 10000)

# Include very small a values so beta can become large near s_beta.
beta_grid = np.geomspace(1e-4, 1e10, 12000)
a_grid = np.concatenate([
    np.array([0.0]),          # beta = infinity, pure truncation
    1.0 / beta_grid,
])
a_grid = np.unique(np.sort(a_grid))
a_grid = np.unique(np.sort(a_grid))

mean_w_ours, second_moment_w_ours = precompute_ours(a_grid, n_nodes=350)
best_a, _ = best_a_over_grid(s_vals, a_grid, mean_w_ours, second_moment_w_ours)

s_beta = (11.0 / 12.0 - math.log(2.0)) / (1.0 - math.log(2.0))

beta_star = np.full_like(s_vals, np.nan, dtype=float)
finite = (s_vals > s_beta) & (best_a > 0.0)
beta_star[finite] = 1.0 / best_a[finite]

fig, ax = plt.subplots(figsize=(3.5, 3))

ax.plot(
    s_vals[finite],
    beta_star[finite],
    color=ACCENT,
    lw=2.2,
    label=r'Optimal finite $\beta$',
)

# ax.fill_between(
#     s_vals,
#     1e-4,
#     np.nan_to_num(beta_star, nan=1e-4),
#     where=finite,
#     color=ACCENT,
#     alpha=0.18,
#     linewidth=0,
#     # label=r'Finite-$\beta$ region',
# )

ax.axvline(s_beta, color='#6E6E6E', lw=1.0, ls='--')
ax.text(
    s_beta + 0.004,
    2.5e-1,
    r'$s_\beta$',
    color='#6E6E6E',
    fontsize=12,
)

ax.set_yscale('log')
ax.set_xlim(0.72, 1.01)
ax.set_ylim(1e-3, 1e4)

ax.set_xlabel(r'Proxy rank where oracle value peaks')
ax.set_ylabel(r'$\beta^\star(s)$')

style_legend(
    ax,
    # ours_label=r'Optimal finite $\beta$',
    fill_edgecolor='#5F6B6C',
    fill_linewidth=0.8,
    # fill_labels={r'Finite-$\beta$ region'},
    corner_radius=0.4,
    loc='upper right',
    # bbox_to_anchor=(1.05, 1.0),
    borderaxespad=0.0,
)

sns.despine(ax=ax)
fig.tight_layout()
fig.savefig('figs/fixed_threshold_beta_star.svg')
plt.show()


## App. B.3: global vs per-prompt truncation

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from paper_style import apply_style, style_legend

# Fixed to the App. B.3 configuration; the arrays come from static_data.npz.
LAMBDA_MODEL = 'llama'
LAMBDA_DATASET = 'ultrafeedback'
LAMBDA_VARIANT = 'random'
LAMBDA_EVAL_SOURCE = 'merged'
LAMBDA_PROXY_RM = 'armorm'
LAMBDA_ORACLE_RM = 'skywork_v2_llama'  # or 'skywork_v2_qwen'
GLOBAL_LAMBDA = 0.5
LAMBDA_BOOTSTRAP_REPETITIONS = 2000
LAMBDA_BOOTSTRAP_SEED = 27

LAMBDA_ORACLE_LABELS = {
    'skywork_v2_llama': 'Skywork-Llama',
    'skywork_v2_qwen': 'Skywork-Qwen',
}


def empirical_win_rate_percentiles(rewards, *, label):
    rewards = np.asarray(rewards, dtype=float)
    if rewards.ndim != 2:
        raise ValueError(f'{label} must be two-dimensional; got shape {rewards.shape}.')
    if rewards.shape[0] == 0 or rewards.shape[1] == 0:
        raise ValueError(f'{label} must be nonempty; got shape {rewards.shape}.')
    if not np.all(np.isfinite(rewards)):
        raise ValueError(f'{label} must contain only finite values.')
    return (rewards[:, None, :] <= rewards[:, :, None]).mean(axis=2)


def analyze_pure_truncation_oracle_gap(
    armorm_rewards,
    oracle_rewards,
    *,
    global_lambda,
):
    if not 0.0 <= global_lambda < 1.0:
        raise ValueError(
            f'GLOBAL_LAMBDA must lie in [0, 1), got {global_lambda!r}.'
        )

    armorm_percentiles = empirical_win_rate_percentiles(
        armorm_rewards,
        label='ArmoRM rewards',
    )
    oracle_percentiles = empirical_win_rate_percentiles(
        oracle_rewards,
        label='Oracle rewards',
    )
    if armorm_percentiles.shape != oracle_percentiles.shape:
        raise ValueError(
            'ArmoRM and oracle rewards must have matching shapes; '
            f'got {armorm_percentiles.shape} and {oracle_percentiles.shape}.'
        )

    num_prompts, num_completions = armorm_percentiles.shape
    threshold_grid = np.arange(num_completions, dtype=float) / num_completions
    utility_by_threshold = np.empty((num_prompts, num_completions), dtype=float)

    for threshold_idx, threshold in enumerate(threshold_grid):
        retained = armorm_percentiles > threshold
        retained_count = retained.sum(axis=1)
        if np.any(retained_count == 0):
            raise RuntimeError(f'Threshold {threshold:g} retained no responses.')
        utility_by_threshold[:, threshold_idx] = (
            (oracle_percentiles * retained).sum(axis=1) / retained_count
        )

    mean_global_utility_by_threshold = utility_by_threshold.mean(axis=0)
    optimal_global_idx = int(np.argmax(mean_global_utility_by_threshold))
    optimal_global_lambda = float(threshold_grid[optimal_global_idx])
    per_prompt_optimal_global_utility = utility_by_threshold[:, optimal_global_idx]
    optimal_global_oracle_utility = float(
        mean_global_utility_by_threshold[optimal_global_idx]
    )

    adaptive_idx = np.argmax(utility_by_threshold, axis=1)
    prompt_idx = np.arange(num_prompts)
    per_prompt_lambda = threshold_grid[adaptive_idx]
    per_prompt_adaptive_utility = utility_by_threshold[prompt_idx, adaptive_idx]
    per_prompt_adaptive_vs_optimal_global_gap = (
        per_prompt_adaptive_utility - per_prompt_optimal_global_utility
    )

    global_retained = armorm_percentiles > global_lambda
    global_retained_count = global_retained.sum(axis=1)
    if np.any(global_retained_count == 0):
        raise RuntimeError('GLOBAL_LAMBDA retained no responses for at least one prompt.')
    per_prompt_global_utility = (
        (oracle_percentiles * global_retained).sum(axis=1)
        / global_retained_count
    )
    per_prompt_gap = per_prompt_adaptive_utility - per_prompt_global_utility

    return {
        'threshold_grid': threshold_grid,
        'utility_by_threshold': utility_by_threshold,
        'mean_global_utility_by_threshold': mean_global_utility_by_threshold,
        'optimal_global_lambda': optimal_global_lambda,
        'per_prompt_optimal_global_utility': per_prompt_optimal_global_utility,
        'optimal_global_oracle_utility': optimal_global_oracle_utility,
        'per_prompt_lambda': per_prompt_lambda,
        'per_prompt_adaptive_utility': per_prompt_adaptive_utility,
        'per_prompt_adaptive_vs_optimal_global_gap': (
            per_prompt_adaptive_vs_optimal_global_gap
        ),
        'per_prompt_global_utility': per_prompt_global_utility,
        'per_prompt_gap': per_prompt_gap,
        'mean_adaptive_utility': float(per_prompt_adaptive_utility.mean()),
        'mean_adaptive_vs_optimal_global_gap': float(
            per_prompt_adaptive_vs_optimal_global_gap.mean()
        ),
        'mean_global_utility': float(per_prompt_global_utility.mean()),
        'mean_gap': float(per_prompt_gap.mean()),
        'num_prompts': num_prompts,
        'num_completions': num_completions,
    }


def bootstrap_mean_confidence_interval(values, *, repetitions, seed):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size == 0 or not np.all(np.isfinite(values)):
        raise ValueError('Bootstrap values must be a nonempty finite one-dimensional array.')
    if (
        isinstance(repetitions, (bool, np.bool_))
        or not isinstance(repetitions, (int, np.integer))
        or repetitions <= 0
    ):
        raise ValueError('Bootstrap repetitions must be a positive integer.')

    rng = np.random.default_rng(seed)
    sample_idx = rng.integers(
        0,
        values.size,
        size=(int(repetitions), values.size),
    )
    bootstrap_means = values[sample_idx].mean(axis=1)
    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


if LAMBDA_PROXY_RM != 'armorm':
    raise ValueError("LAMBDA_PROXY_RM is fixed to 'armorm'.")
if LAMBDA_ORACLE_RM not in LAMBDA_ORACLE_LABELS:
    raise ValueError(
        f'LAMBDA_ORACLE_RM must be one of {list(LAMBDA_ORACLE_LABELS)}, '
        f'got {LAMBDA_ORACLE_RM!r}.'
    )
if not 0.0 <= GLOBAL_LAMBDA < 1.0:
    raise ValueError(f'GLOBAL_LAMBDA must lie in [0, 1), got {GLOBAL_LAMBDA!r}.')

lambda_rewards = {rm: _npz[f"ultrafeedback/random/merged/{rm}"] for rm in (LAMBDA_PROXY_RM, LAMBDA_ORACLE_RM)}
lambda_oracle_gap = analyze_pure_truncation_oracle_gap(
    lambda_rewards[LAMBDA_PROXY_RM],
    lambda_rewards[LAMBDA_ORACLE_RM],
    global_lambda=GLOBAL_LAMBDA,
)
oracle_gap_ci = bootstrap_mean_confidence_interval(
    lambda_oracle_gap['per_prompt_gap'],
    repetitions=LAMBDA_BOOTSTRAP_REPETITIONS,
    seed=LAMBDA_BOOTSTRAP_SEED,
)

per_prompt_lambdas = lambda_oracle_gap['per_prompt_lambda']
mean_per_prompt_lambda = float(per_prompt_lambdas.mean())
lambda_values, lambda_counts = np.unique(per_prompt_lambdas, return_counts=True)
lambda_probability = lambda_counts / per_prompt_lambdas.size
num_completions = lambda_oracle_gap['num_completions']

lambda_oracle_gap_summary = pd.DataFrame([
    {
        'Prompts': lambda_oracle_gap['num_prompts'],
        'Refs / prompt': num_completions,
        'Global lambda': GLOBAL_LAMBDA,
        'Adaptive oracle utility': lambda_oracle_gap['mean_adaptive_utility'],
        'Global oracle utility': lambda_oracle_gap['mean_global_utility'],
        'Optimal global lambda (same-pool)': (
            lambda_oracle_gap['optimal_global_lambda']
        ),
        'Optimal global oracle utility': (
            lambda_oracle_gap['optimal_global_oracle_utility']
        ),
        'Adaptive - optimal global (pp)': (
            100.0 * lambda_oracle_gap['mean_adaptive_vs_optimal_global_gap']
        ),
        'Oracle gap': lambda_oracle_gap['mean_gap'],
        'Gap (percentile points)': 100.0 * lambda_oracle_gap['mean_gap'],
        '95% bootstrap CI (pp)': (
            f'[{100.0 * oracle_gap_ci[0]:.2f}, '
            f'{100.0 * oracle_gap_ci[1]:.2f}]'
        ),
    }
])
display(lambda_oracle_gap_summary.round({
    'Global lambda': 3,
    'Adaptive oracle utility': 4,
    'Global oracle utility': 4,
    'Optimal global lambda (same-pool)': 3,
    'Optimal global oracle utility': 4,
    'Adaptive - optimal global (pp)': 2,
    'Oracle gap': 4,
    'Gap (percentile points)': 2,
}))

apply_style()
fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.bar(
    lambda_values,
    lambda_probability,
    width=0.8 / num_completions,
    color='#8AA4B0',
    edgecolor='white',
    linewidth=0.4,
    label='Prompt-wise utility-maximizing thresholds',
)
ax.axvline(
    mean_per_prompt_lambda,
    color='#B84040',
    lw=1.5,
    ls='--',
    label=rf'Mean $={mean_per_prompt_lambda:.2f}$',
)
ax.axvline(
    GLOBAL_LAMBDA,
    color='#4F6F52',
    lw=1.5,
    ls=':',
    label=rf'Global $\lambda={GLOBAL_LAMBDA:g}$',
)
ax.axvline(
    lambda_oracle_gap['optimal_global_lambda'],
    color='#6F4E7C',
    lw=1.5,
    ls='-',
    label=(
        'Optimal global (same-pool) '
        rf'$={lambda_oracle_gap["optimal_global_lambda"]:.2f}$'
    ),
)
ax.set_xlabel(r'Prompt-wise optimal threshold $\lambda_i^*$')
ax.set_ylabel('Fraction of prompts')
ax.set_title(
    f'Llama + UltraFeedback (random, merged): ArmoRM proxy, '
    f'{LAMBDA_ORACLE_LABELS[LAMBDA_ORACLE_RM]} oracle'
)
ax.set_xlim(-0.02, 1.0)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
style_legend(ax, fontsize=8, loc='upper left', corner_radius=0.8)
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

fig_utility, ax_utility = plt.subplots(figsize=(5.2, 3.0))
ax_utility.plot(
    lambda_oracle_gap['threshold_grid'],
    lambda_oracle_gap['mean_global_utility_by_threshold'],
    color='#3D6173',
    lw=1.8,
)
ax_utility.axvline(
    lambda_oracle_gap['optimal_global_lambda'],
    color='#6F4E7C',
    lw=1.5,
    ls='-',
    label=(
        'Optimal global (same-pool) '
        rf'$\lambda={lambda_oracle_gap["optimal_global_lambda"]:.2f}$'
    ),
)
ax_utility.axvline(
    GLOBAL_LAMBDA,
    color='#4F6F52',
    lw=1.5,
    ls=':',
    label=rf'Configured global $\lambda={GLOBAL_LAMBDA:g}$',
)
ax_utility.set_xlabel(r'Global truncation threshold $\lambda$')
ax_utility.set_ylabel('Mean oracle win-rate percentile')
ax_utility.set_title(
    f'Llama + UltraFeedback (random, merged): same-pool global search, '
    f'{LAMBDA_ORACLE_LABELS[LAMBDA_ORACLE_RM]} oracle'
)
ax_utility.set_xlim(-0.02, 1.0)
ax_utility.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
style_legend(ax_utility, fontsize=8, loc='best', corner_radius=0.8)
sns.despine(ax=ax_utility)
fig_utility.tight_layout()
plt.show()
